# Notebook 03: Lists — Ordered Collections in Redis

Redis Lists are **linked lists** of string values. They're:
- **Ordered** — elements keep their insertion order
- **Allow duplicates** — unlike Sets
- **O(1) push/pop** at both ends — very fast!
- **O(n) access by index** — slower for middle elements

Think of them like a **Python list**, but shared across applications and stored in memory.

**Common uses:** Task queues, activity feeds, chat history, undo stacks, recent items.

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## 1. LPUSH and RPUSH — Adding Elements

Lists have two ends: **Left (head)** and **Right (tail)**.

```
LPUSH adds here →  [  A  |  B  |  C  |  D  ]  ← RPUSH adds here
                    head                 tail
                    index 0              index -1
```

In [ ]:
# RPUSH — Add to the RIGHT (tail/end)
# Redis CLI: RPUSH fruits "apple" "banana" "cherry"
r.rpush('fruits', 'apple', 'banana', 'cherry')

# LRANGE — Get elements by range (0 to -1 means ALL)
# Redis CLI: LRANGE fruits 0 -1
print(f"After RPUSH: {r.lrange('fruits', 0, -1)}")
# → ['apple', 'banana', 'cherry'] — in order!

In [ ]:
# LPUSH — Add to the LEFT (head/beginning)
# Redis CLI: LPUSH fruits "mango" "grape"
r.lpush('fruits', 'mango', 'grape')

print(f"After LPUSH: {r.lrange('fruits', 0, -1)}")
# → ['grape', 'mango', 'apple', 'banana', 'cherry']
# Notice: 'grape' is first because LPUSH pushes each element to the left
# 'mango' was pushed first, then 'grape' was pushed to the left of 'mango'

> **Important:** When LPUSH gets multiple values, it pushes them one by one from left to right. So `LPUSH list a b c` results in `[c, b, a, ...]` — they end up in reverse order!

---
## 2. LRANGE — Getting Elements

Works like Python slicing. Negative indices count from the end.

In [ ]:
r.delete('numbers')
r.rpush('numbers', 'one', 'two', 'three', 'four', 'five')
print(f"Full list: {r.lrange('numbers', 0, -1)}")

# First 3 elements
print(f"First 3:   {r.lrange('numbers', 0, 2)}")

# Last 2 elements
print(f"Last 2:    {r.lrange('numbers', -2, -1)}")

# Middle elements
print(f"Index 1-3: {r.lrange('numbers', 1, 3)}")

---
## 3. LPOP and RPOP — Removing Elements

In [ ]:
r.delete('tasks')
r.rpush('tasks', 'email', 'report', 'meeting', 'code', 'review')
print(f"Tasks: {r.lrange('tasks', 0, -1)}")

# LPOP — Remove and return from the LEFT (first element)
# Redis CLI: LPOP tasks
first = r.lpop('tasks')
print(f"\nLPOP returned: '{first}'")
print(f"Remaining: {r.lrange('tasks', 0, -1)}")

# RPOP — Remove and return from the RIGHT (last element)
# Redis CLI: RPOP tasks
last = r.rpop('tasks')
print(f"\nRPOP returned: '{last}'")
print(f"Remaining: {r.lrange('tasks', 0, -1)}")

In [ ]:
# Pop multiple elements at once (Redis 6.2+)
r.delete('items')
r.rpush('items', 'a', 'b', 'c', 'd', 'e')

popped = r.lpop('items', 3)  # Pop 3 from the left
print(f"Popped 3: {popped}")
print(f"Remaining: {r.lrange('items', 0, -1)}")

---
## 4. LLEN, LINDEX, LSET

In [ ]:
r.delete('colors')
r.rpush('colors', 'red', 'green', 'blue', 'yellow', 'purple')

# LLEN — Get list length
# Redis CLI: LLEN colors
print(f"Length: {r.llen('colors')}")

# LINDEX — Get element by index (0-based)
# Redis CLI: LINDEX colors 0
print(f"Index 0 (first): {r.lindex('colors', 0)}")
print(f"Index -1 (last): {r.lindex('colors', -1)}")
print(f"Index 2: {r.lindex('colors', 2)}")

# LSET — Set element at index
# Redis CLI: LSET colors 2 "cyan"
r.lset('colors', 2, 'cyan')  # Replace 'blue' with 'cyan'
print(f"After LSET: {r.lrange('colors', 0, -1)}")

---
## 5. LINSERT — Insert Before/After a Value

In [ ]:
r.delete('playlist')
r.rpush('playlist', 'song_A', 'song_B', 'song_D')
print(f"Before: {r.lrange('playlist', 0, -1)}")

# Insert 'song_C' BEFORE 'song_D'
# Redis CLI: LINSERT playlist BEFORE "song_D" "song_C"
r.linsert('playlist', 'BEFORE', 'song_D', 'song_C')
print(f"After insert before: {r.lrange('playlist', 0, -1)}")

# Insert 'song_E' AFTER 'song_D'
r.linsert('playlist', 'AFTER', 'song_D', 'song_E')
print(f"After insert after:  {r.lrange('playlist', 0, -1)}")

---
## 6. LREM — Remove Specific Elements

`LREM key count value` — The `count` parameter controls behavior:
- `count > 0`: Remove `count` occurrences from HEAD (left to right)
- `count < 0`: Remove `|count|` occurrences from TAIL (right to left)
- `count = 0`: Remove ALL occurrences

In [ ]:
r.delete('letters')
r.rpush('letters', 'a', 'b', 'a', 'c', 'a', 'b', 'a')
print(f"Original: {r.lrange('letters', 0, -1)}")

# Remove first 2 occurrences of 'a' from the left
# Redis CLI: LREM letters 2 "a"
removed = r.lrem('letters', 2, 'a')
print(f"Removed {removed} 'a's from left: {r.lrange('letters', 0, -1)}")

# Remove ALL remaining 'a's
removed = r.lrem('letters', 0, 'a')
print(f"Removed {removed} 'a's (all): {r.lrange('letters', 0, -1)}")

---
## 7. LTRIM — Keep Only a Range (Trim the Rest)

This is extremely useful for keeping only the **N most recent** items.

In [ ]:
# Imagine a log that should only keep the last 5 entries
r.delete('recent_logs')

for i in range(10):
    r.lpush('recent_logs', f'log_entry_{i}')

print(f"All 10 logs: {r.lrange('recent_logs', 0, -1)}")
print(f"Length: {r.llen('recent_logs')}")

# Keep only the 5 most recent (indices 0-4)
# Redis CLI: LTRIM recent_logs 0 4
r.ltrim('recent_logs', 0, 4)

print(f"\nAfter LTRIM (keep 5): {r.lrange('recent_logs', 0, -1)}")
print(f"Length: {r.llen('recent_logs')}")

---
## 8. LPOS — Find Position of an Element

In [ ]:
r.delete('names')
r.rpush('names', 'alice', 'bob', 'charlie', 'bob', 'diana')

# Redis CLI: LPOS names "charlie"
pos = r.lpos('names', 'charlie')
print(f"'charlie' is at index: {pos}")

pos = r.lpos('names', 'bob')
print(f"First 'bob' is at index: {pos}")

# Key doesn't exist returns None
pos = r.lpos('names', 'unknown')
print(f"'unknown' position: {pos}")

---
## 9. LMOVE — Atomically Move Elements Between Lists

Move an element from one list to another in a single atomic operation. This is the foundation of **reliable queues**.

In [ ]:
# Reliable queue pattern: move task from 'pending' to 'processing'
r.delete('pending', 'processing')

r.rpush('pending', 'task_1', 'task_2', 'task_3')
print(f"Pending:    {r.lrange('pending', 0, -1)}")
print(f"Processing: {r.lrange('processing', 0, -1)}")

# Move first pending task to processing
# Redis CLI: LMOVE pending processing LEFT RIGHT
task = r.lmove('pending', 'processing', 'LEFT', 'RIGHT')
print(f"\nMoved: '{task}'")
print(f"Pending:    {r.lrange('pending', 0, -1)}")
print(f"Processing: {r.lrange('processing', 0, -1)}")

---
## 10. BLPOP / BRPOP — Blocking Pop (Wait for Data)

These are like LPOP/RPOP but they **wait** if the list is empty. This turns a list into a **real-time message queue**.

> Note: In a notebook we need to use a timeout so it doesn't hang forever.

In [ ]:
import threading

r.delete('job_queue')

def worker():
    """Worker that waits for jobs using BLPOP."""
    worker_r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    print("  Worker: Waiting for a job...")
    result = worker_r.blpop('job_queue', timeout=5)  # Wait up to 5 seconds
    if result:
        list_name, value = result
        print(f"  Worker: Got job '{value}' from '{list_name}'!")
    else:
        print("  Worker: Timed out, no job received.")

# Start worker in a thread (it will block waiting for data)
t = threading.Thread(target=worker, daemon=True)
t.start()

# Wait a moment, then push a job
time.sleep(1)
print("Producer: Pushing a job...")
r.rpush('job_queue', 'process_report_42')

t.join(timeout=3)
print("Done!")

---
## 11. Real-World: Simple Task Queue

A producer adds tasks, a consumer processes them — the classic queue pattern.

In [ ]:
r.delete('task_queue', 'completed')

# Producer: Add tasks to the queue
tasks = ['send_email_user1', 'generate_report', 'resize_image_42', 'send_notification']
for task in tasks:
    r.rpush('task_queue', task)
    print(f"  Produced: {task}")

print(f"\nQueue length: {r.llen('task_queue')}")
print("---")

# Consumer: Process tasks one by one
while r.llen('task_queue') > 0:
    task = r.lpop('task_queue')  # Get the oldest task
    print(f"  Processing: {task}...")
    time.sleep(0.3)  # Simulate work
    r.rpush('completed', task)  # Mark as done
    print(f"  Completed: {task}")

print(f"\nAll done! Completed: {r.lrange('completed', 0, -1)}")

---
## 12. Real-World: Activity Feed (Recent N Items)

Keep only the last 10 activities for each user using `LPUSH` + `LTRIM`.

In [ ]:
def add_activity(user_id, activity, max_items=5):
    """Add an activity and keep only the most recent N."""
    key = f"activity:{user_id}"
    r.lpush(key, activity)       # Add to the front (most recent first)
    r.ltrim(key, 0, max_items - 1)  # Keep only last N

def get_activity(user_id):
    """Get all recent activities for a user."""
    return r.lrange(f"activity:{user_id}", 0, -1)

r.delete('activity:sujit')

# Sujit does a bunch of things
activities = [
    'Logged in',
    'Viewed dashboard',
    'Updated profile',
    'Uploaded photo',
    'Commented on post',
    'Liked a photo',
    'Sent a message',
    'Changed password'
]

for act in activities:
    add_activity('sujit', act, max_items=5)

print("Recent activities (most recent first):")
for i, act in enumerate(get_activity('sujit'), 1):
    print(f"  {i}. {act}")

print(f"\nOnly 5 stored, even though 8 activities happened!")

---
## 13. Real-World: Undo History (Stack)

Use a list as a **stack** (LIFO — Last In, First Out) for undo functionality.

In [ ]:
r.delete('undo:doc1')

def perform_action(doc_id, action):
    """Perform an action and record it for undo."""
    r.lpush(f'undo:{doc_id}', action)  # Push to stack
    print(f"  Action: {action}")

def undo(doc_id):
    """Undo the last action."""
    action = r.lpop(f'undo:{doc_id}')  # Pop from stack
    if action:
        print(f"  Undo: {action}")
    else:
        print("  Nothing to undo!")

# User edits a document
perform_action('doc1', 'typed: Hello')
perform_action('doc1', 'typed: World')
perform_action('doc1', 'made text bold')
perform_action('doc1', 'changed font to Arial')

print("\n--- Undoing ---")
undo('doc1')  # Undo 'changed font to Arial'
undo('doc1')  # Undo 'made text bold'

print(f"\nRemaining undo history: {r.lrange('undo:doc1', 0, -1)}")

---
## Lists vs Python Lists

| Feature | Python List | Redis List |
|---|---|---|
| Storage | In your program's memory | In Redis server (shared) |
| Access by index | O(1) — instant | O(n) — slower for middle |
| Push/Pop at ends | O(1) amortized | O(1) — always fast |
| Shared access | No (one program) | Yes (any client) |
| Persistence | Lost when program exits | Can persist to disk |
| Max size | Limited by RAM | Limited by Redis maxmemory |
| Blocking pop | No built-in | BLPOP/BRPOP — wait for data |

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

```
RPUSH key val1 val2    → Add to right (tail)
LPUSH key val1 val2    → Add to left (head)
LRANGE key 0 -1        → Get all elements
LPOP key               → Remove from left
RPOP key               → Remove from right
LLEN key               → Get length
LINDEX key n           → Get element at index n
LSET key n val         → Set element at index n
LINSERT key BEFORE/AFTER pivot val → Insert near a value
LREM key count val     → Remove occurrences
LTRIM key start stop   → Keep only range
LMOVE src dst LEFT/RIGHT LEFT/RIGHT → Move between lists
BLPOP key timeout      → Blocking pop (wait for data)
```

### Key Patterns
- **Queue (FIFO):** RPUSH to add, LPOP to consume
- **Stack (LIFO):** LPUSH to add, LPOP to undo
- **Recent N items:** LPUSH + LTRIM to cap the list
- **Reliable queue:** LMOVE from pending to processing

---
## Exercises

1. **Browser History:** Implement a browser history using a list. Add 10 URLs, then implement "back" (pop the current URL to reveal the previous one). Print the history at each step.

2. **Bounded Queue:** Create a function `enqueue(queue_name, item, max_size)` that adds an item but keeps the queue at max `max_size` items. Test it with a max of 3.

3. **Priority Processing:** Create two queues: `urgent` and `normal`. Write a consumer that always processes `urgent` tasks first (check with LLEN), and only processes `normal` tasks when `urgent` is empty.

4. **Circular Buffer:** Implement a circular buffer using RPUSH and LTRIM that keeps exactly the last N items. Push 20 items but keep only the last 7. Verify the result.

5. **Chat Room:** Use RPUSH to add messages to a chat room list. Each message should include the username and timestamp. Write a function to get the last N messages.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 04 — Sets & Sorted Sets](./04_Sets_and_Sorted_Sets.ipynb)** — Unique collections, set operations (union, intersection), and building leaderboards!